In [3]:
#day3：缺失值、重复值、异常值 目标：掌握数据清洗

In [4]:
import pandas as pd
import os
import glob
import sqlite3

from fontTools.merge import tables
from pandas.core.algorithms import duplicated

#读取数据
data_path = './data/'
#读取所有数据
csv_files = glob.glob(data_path + '*.csv')
#建立字典
dataframes = {}
#遍历数据
for file in csv_files:
    name = os.path.basename(file).replace('.csv', '')  #替换将后缀替换为空
    dataframes[name] = pd.read_csv(file)


orders= dataframes[('olist_orders_dataset')]
conn = sqlite3.connect('olist.db')

In [5]:
#查询
orders = dataframes["olist_orders_dataset"]
missing=orders.isnull().sum()
missing

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [28]:
query = """
SELECT
order_status,
COUNT(*) as total_orders,
SUM(
CASE WHEN order_approved_at IS NULL
THEN 1 ELSE 0 END
)AS missing_approver,
SUM(
CASE WHEN order_delivered_carrier_date IS NULL
THEN 1 ELSE 0 END
)AS missing_delivered_carrier,
SUM(
CASE WHEN
 order_delivered_customer_date IS NULL
THEN 1 ELSE 0 END
) AS missing_customer
FROM olist_orders_dataset
GROUP BY order_status

"""

result = pd.read_sql_query(query,conn)
result

,order_status,total_orders,missing_approver,missing_delivered_carrier,missing_customer
0,approved,2,0,2,2
1,canceled,625,141,550,619
2,created,5,5,5,5
3,delivered,96478,14,2,8
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,shipped,1107,0,0,1107
7,unavailable,609,0,609,609


In [22]:
#筛选异常订单
query = """
SELECT
order_id,
order_status,
order_approved_at,
order_delivered_customer_date,
order_delivered_carrier_date
FROM olist_orders_dataset
WHERE order_status = 'delivered'
AND (
order_approved_at IS NULL
OR order_delivered_carrier_date IS NULL
OR order_delivered_customer_date IS NULL
) ;
"""
result = pd.read_sql_query(query,conn)
result

,order_id,order_status,order_approved_at,order_delivered_customer_date,order_delivered_carrier_date
0,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:56:40,NaN,2017-11-30 18:12:23
1,e04abd8149ef81b95221e88f6ed9ab6a,delivered,NaN,2017-03-01 13:25:33,2017-02-23 12:04:47
2,8a9adc69528e1001fc68dd0aaebbb54a,delivered,NaN,2017-03-02 10:05:06,2017-02-23 09:01:52
3,7013bcfc1c97fe719a7b5e05e61c12db,delivered,NaN,2017-03-01 08:07:38,2017-02-22 16:25:25
4,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 07:19:05,NaN,2018-06-25 08:05:00
5,5cf925b116421afa85ee25e99b4c34fb,delivered,NaN,2017-03-09 07:28:47,2017-02-22 11:23:10
6,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,NaN,2017-03-02 11:09:19,2017-02-22 11:23:11
7,c1d4211b3dae76144deccd6c74144a88,delivered,NaN,2017-01-30 18:16:01,2017-01-25 14:56:50
8,d69e5d356402adc8cf17e08b5033acfb,delivered,NaN,2017-03-02 03:41:58,2017-02-23 03:11:48
9,d77031d6a3c8a52f019764e68f211c69,delivered,NaN,2017-03-02 16:15:23,2017-02-23 07:23:36


In [8]:
#检查时间先后顺序,发货时间晚于收货时间
query = """
SELECT
order_id,
order_delivered_carrier_date,
order_delivered_customer_date
FROM olist_orders_dataset
WHERE order_delivered_carrier_date IS NOT NULL
AND order_delivered_customer_date IS NOT NULL
AND order_delivered_customer_date < order_delivered_carrier_date;
"""
result = pd.read_sql_query(query,conn)
result
#无显示，证明没有异常

,order_id,order_delivered_carrier_date,order_delivered_customer_date
0,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56
1,383aa8b2724fe452d9ccd9934a8c628b,2017-07-07 17:22:41,2017-07-06 14:27:51
2,cb1134f9010d242e9515ad1c78ec0c39,2017-07-20 19:22:02,2017-07-19 14:13:28
3,dceb62e8fa94b46006c9554fed743df0,2017-08-01 18:23:30,2017-07-26 18:09:10
4,5f9d46795c3126674e52becb3a1a517f,2017-07-20 23:03:42,2017-07-20 18:52:41
5,8c78d01de3a9009e23d6877a7cc9be20,2016-10-26 11:41:53,2016-10-25 17:51:46
6,b27af682321527a6349f1761eb3f360c,2017-06-27 14:51:54,2017-06-26 15:45:35
7,1cc3ae63caffff2d6c3ee3e78e074acf,2017-08-10 18:28:56,2017-08-10 18:05:38
8,e37f11cae9985ca58f0b56f268720537,2017-08-01 18:17:47,2017-07-31 17:49:56
9,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-08-09 18:18:43,2017-08-01 21:13:01


In [9]:
#订单查重
orders = dataframes["olist_orders_dataset"]
print("重复单号数量",orders["order_id"].duplicated().sum())#检查id重复
print("完全重复记录数量",orders.duplicated().sum())#检查行重复

重复单号数量 0
完全重复记录数量 0


In [35]:
#明细查重
items = dataframes["olist_order_items_dataset"]
print("完全重复记录数量",items.duplicated().sum())
print("订单编号重复数量",items["order_id"].duplicated().sum())
print("联合主键重复记录",items.duplicated(subset=["order_id","order_item_id"]).sum())

完全重复记录数量 0
订单编号重复数量 13984
联合主键重复记录 0


In [11]:
print(dataframes.keys())#数据库的表名

dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])


In [12]:
tables =[
    "olist_order_payments_dataset",
    "olist_customers_dataset",
    "olist_geolocation_dataset",#地理位置
    "olist_order_reviews_dataset"
]
for name in tables:
    df = dataframes[name]
    print(
        name,
        "完全重复",df.duplicated().sum()
    )

olist_order_payments_dataset 完全重复 0
olist_customers_dataset 完全重复 0
olist_geolocation_dataset 完全重复 261831
olist_order_reviews_dataset 完全重复 0


In [13]:
geo = dataframes["olist_geolocation_dataset"]
print(geo.head())
print(geo.columns.tolist())

   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   
2                         1046       -23.546129       -46.642951   
3                         1041       -23.544392       -46.639499   
4                         1035       -23.541578       -46.641607   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


In [14]:
print("邮政编码重复数量：",
      geo["geolocation_zip_code_prefix"].duplicated().sum())
print("完全重复数量",geo.duplicated().sum())

邮政编码重复数量： 981148
完全重复数量 261831


In [15]:
#查看统计重复记录
duplicated_geo = geo[geo.duplicated(keep=False)]#keep=false,第二次出现的数据，也就是重复数据,并标注所有重复行
print(duplicated_geo.head(10))

    geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                          1037       -23.545621       -46.639292   
1                          1046       -23.546081       -46.644820   
2                          1046       -23.546129       -46.642951   
6                          1047       -23.546273       -46.641225   
7                          1013       -23.546923       -46.634264   
8                          1029       -23.543769       -46.634278   
9                          1011       -23.547640       -46.636032   
10                         1013       -23.547325       -46.634184   
13                         1012       -23.548946       -46.634671   
15                         1046       -23.546081       -46.644820   

   geolocation_city geolocation_state  
0         sao paulo                SP  
1         sao paulo                SP  
2         sao paulo                SP  
6         sao paulo                SP  
7         sao paulo                SP  


In [16]:
print("原始记录数",len(geo))
print("重复记录筛选后：",len(duplicated_geo))

原始记录数 1000163
重复记录筛选后： 390005


In [17]:
#完全重复有261831个，全部重复有390005个，也就是说，有128174个数据是第一次出现的
print(duplicated_geo.head(10))

    geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                          1037       -23.545621       -46.639292   
1                          1046       -23.546081       -46.644820   
2                          1046       -23.546129       -46.642951   
6                          1047       -23.546273       -46.641225   
7                          1013       -23.546923       -46.634264   
8                          1029       -23.543769       -46.634278   
9                          1011       -23.547640       -46.636032   
10                         1013       -23.547325       -46.634184   
13                         1012       -23.548946       -46.634671   
15                         1046       -23.546081       -46.644820   

   geolocation_city geolocation_state  
0         sao paulo                SP  
1         sao paulo                SP  
2         sao paulo                SP  
6         sao paulo                SP  
7         sao paulo                SP  


In [18]:
exact_duplicates = geo[geo.duplicated(keep="first")]
print("完全重复记录数量",len(exact_duplicates))
print(exact_duplicates.head(10))

完全重复记录数量 261831
    geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
15                         1046       -23.546081       -46.644820   
44                         1046       -23.546081       -46.644820   
65                         1046       -23.546081       -46.644820   
66                         1009       -23.546935       -46.636588   
67                         1046       -23.546081       -46.644820   
72                         1046       -23.545320       -46.644069   
79                         1050       -23.549854       -46.643139   
80                         1032       -23.540775       -46.635515   
82                         1046       -23.546081       -46.644820   
86                         1048       -23.547449       -46.640169   

   geolocation_city geolocation_state  
15        sao paulo                SP  
44        sao paulo                SP  
65        sao paulo                SP  
66        sao paulo                SP  
67        sao paulo     

In [19]:
print("默认重复数量：", geo.duplicated().sum())#除去第一次出现，标记以后
print("keep='first'：", geo.duplicated(keep="first").sum())#第一次出现2，3...x
print("keep='last'：", geo.duplicated(keep="last").sum())#最后一次出现1.2..x-1
print("keep=False：", geo.duplicated(keep=False).sum())#统计全部，1.2.3....x

默认重复数量： 261831
keep='first'： 261831
keep='last'： 261831
keep=False： 390005


In [20]:
#生成去重副本，方便回滚
geo_clean = geo.drop_duplicates()
print("原始记录数",len(geo))
print("去重后记录数",len(geo_clean))
print("删除记录数",len(geo)-len(geo_clean))

原始记录数 1000163
去重后记录数 738332
删除记录数 261831
